# T2.2 — Multilingual Grant & Tender Matcher
## Evaluation Notebook: MRR@5, Recall@5, Confusion Analysis

**Challenge:** AIMS KTT Hackathon · T2.2  
**Date:** April 2026

---

This notebook:
1. Generates data (if not already present)
2. Runs the matcher across all 10 profiles
3. Computes MRR@5 and Recall@5 vs. `gold_matches.csv`
4. Displays per-profile score breakdown
5. Shows 3 confusion cases with root-cause analysis

In [ ]:
# ── 0. Install dependencies if needed ─────────────────────────────────────────
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'sentence-transformers', 'scikit-learn', 'langdetect',
    'beautifulsoup4', 'pdfplumber', 'pandas', 'matplotlib', 'seaborn'])
print('✅ Dependencies ready')

In [ ]:
# ── 1. Generate data ───────────────────────────────────────────────────────────
import os
if not os.path.exists('profiles.json') or not os.path.exists('tenders'):
    print('Generating synthetic data …')
    exec(open('generate_data.py').read())
else:
    print('✅ Data already present')

In [ ]:
# ── 2. Run evaluation ──────────────────────────────────────────────────────────
from evaluate import evaluate, find_confusion_cases

results, metrics = evaluate(topk=5)

print('\n📊 Global Metrics')
print(f"  MRR@5    : {metrics['MRR@5']}")
print(f"  Recall@5 : {metrics['Recall@5']}")
print(f"  Profiles : {metrics['n_profiles']}")

In [ ]:
# ── 3. Per-profile results table ───────────────────────────────────────────────
import pandas as pd

rows = []
for r in results:
    rows.append({
        'Profile': r['profile_id'],
        'Sector': r['sector'],
        'Country': r['country'],
        'Gold Matches': ', '.join(r['relevant']),
        'Top Predicted': r['predicted'][0] if r['predicted'] else '',
        'Hits': r['hits'],
        'RR': r['rr'],
        'Recall@5': r['recall_at_k'],
    })

df = pd.DataFrame(rows)
display(df.style
    .background_gradient(subset=['RR', 'Recall@5'], cmap='RdYlGn')
    .format({'RR': '{:.2f}', 'Recall@5': '{:.2f}'}))

In [ ]:
# ── 4. Bar chart: MRR and Recall per profile ───────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

profile_ids = df['Profile'].tolist()
x = np.arange(len(profile_ids))
width = 0.6

# MRR
ax = axes[0]
colors = ['#2ecc71' if v >= 0.5 else '#e74c3c' for v in df['RR']]
bars = ax.bar(x, df['RR'], width, color=colors, edgecolor='white', linewidth=0.5)
ax.set_xticks(x)
ax.set_xticklabels([f'P{p}' for p in profile_ids])
ax.set_ylim(0, 1.1)
ax.set_ylabel('Reciprocal Rank')
ax.set_title('MRR@5 per Profile', fontsize=13, fontweight='bold')
ax.axhline(metrics['MRR@5'], color='#2c3e50', linestyle='--', linewidth=1.5,
           label=f"Mean = {metrics['MRR@5']}")
ax.legend()
ax.bar_label(bars, fmt='%.2f', fontsize=8, padding=2)

# Recall
ax = axes[1]
colors2 = ['#3498db' if v >= 0.5 else '#e67e22' for v in df['Recall@5']]
bars2 = ax.bar(x, df['Recall@5'], width, color=colors2, edgecolor='white', linewidth=0.5)
ax.set_xticks(x)
ax.set_xticklabels([f'P{p}' for p in profile_ids])
ax.set_ylim(0, 1.1)
ax.set_ylabel('Recall')
ax.set_title('Recall@5 per Profile', fontsize=13, fontweight='bold')
ax.axhline(metrics['Recall@5'], color='#2c3e50', linestyle='--', linewidth=1.5,
           label=f"Mean = {metrics['Recall@5']}")
ax.legend()
ax.bar_label(bars2, fmt='%.2f', fontsize=8, padding=2)

plt.suptitle('T2.2 — Multilingual Tender Matcher Evaluation', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('evaluation_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved → evaluation_metrics.png')

In [ ]:
# ── 5. Score component heatmap ─────────────────────────────────────────────────
import seaborn as sns

# Collect score breakdowns for top-1 prediction per profile
heatmap_data = []
for r in results:
    if r['ranked_objects']:
        bd = r['ranked_objects'][0].get('score_breakdown', {})
        row = {'Profile': f"P{r['profile_id']} ({r['sector'][:6]})", **bd}
        heatmap_data.append(row)

hdf = pd.DataFrame(heatmap_data).set_index('Profile')

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(hdf, annot=True, fmt='.2f', cmap='YlOrRd', ax=ax,
            linewidths=0.5, linecolor='white', cbar_kws={'label': 'Score'})
ax.set_title('Score Component Breakdown — Top-1 Prediction per Profile',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Scoring Component')
plt.tight_layout()
plt.savefig('score_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Heatmap saved → score_heatmap.png')

In [ ]:
# ── 6. Confusion Cases Analysis ────────────────────────────────────────────────
from evaluate import find_confusion_cases

cases = find_confusion_cases(results, n=3)

print('⚠️  Top 3 Confusion Cases\n')
print('=' * 70)

for i, c in enumerate(cases, 1):
    print(f"\n{'─'*70}")
    print(f"Case {i} — Profile {c['profile_id']} | Sector: {c['sector']} | Country: N/A")
    print(f"  RR = {c['rr']} | Recall = {c['recall']}")
    print(f"  Top Predicted : {c['top_predicted']} (score={c['top_score']:.4f}, sector={c['top_sector']}")
    print(f"  Gold Answers  : {c['gold_ids']}")
    print(f"  Budget of top : {c['top_budget']}")
    print()
    print(f"  Score Breakdown:")
    for k, v in c['breakdown'].items():
        bar = '█' * int(v * 20) + '░' * (20 - int(v * 20))
        print(f"    {k:15s}: {v:.2f}  {bar}")
    print()

    # Root cause
    bd = c['breakdown']
    if c['top_sector'] != c['sector']:
        print(f"  🔍 ROOT CAUSE: Sector mismatch.")
        print(f"     The tender sector '{c['top_sector']}' ≠ profile sector '{c['sector']}'.")
        print(f"     High semantic similarity ({bd.get('semantic',0):.2f}) inflated the score")
        print(f"     despite low sector_match ({bd.get('sector_match',0):.2f}).")
        print(f"  💡 FIX: Increase sector_match weight from 0.25 → 0.35 for this sector pair.")
    elif bd.get('deadline', 1.0) < 0.3:
        print(f"  🔍 ROOT CAUSE: Near-expired deadline ({bd.get('deadline',0):.2f}).")
        print(f"  💡 FIX: Add a hard filter — exclude tenders with < 7 days remaining.")
    else:
        print(f"  🔍 ROOT CAUSE: Budget mismatch or multi-factor overlap.")
        print(f"     Budget fit = {bd.get('budget_fit',0):.2f} — tender may be too large/small.")
        print(f"  💡 FIX: Add employee-count-based budget ceiling to budget_fit_score().")

print(f"\n{'='*70}")

In [ ]:
# ── 7. Summary Statistics Table ────────────────────────────────────────────────
summary_stats = pd.DataFrame({
    'Metric': ['MRR@5', 'Recall@5', 'Profiles Evaluated', 'Tenders Corpus',
               'EN Tenders', 'FR Tenders', 'Avg Score (Top-1)', 'Model'],
    'Value': [
        metrics['MRR@5'],
        metrics['Recall@5'],
        metrics['n_profiles'],
        40,
        32,
        8,
        round(sum(r['ranked_objects'][0]['score'] for r in results if r['ranked_objects']) / len(results), 4),
        'paraphrase-multilingual-MiniLM-L12-v2'
    ]
})

print('\n📋 Summary Statistics')
display(summary_stats)
summary_stats.to_csv('evaluation_summary.csv', index=False)
print('Saved → evaluation_summary.csv')